# Neural network training, one leaflet for training and the other for testing

For each chain (sn1, sn2) and each window (raw, 3 pt, …):

1. **Train** on the `TRAIN_LEAFLET` table (default lower), **test** on the `TEST_LEAFLET` table (default upper).
   Both come from `training_df_generation` (`training_{chain}_{leaflet}_ft_{window}.csv`).
2. The scaler is fitted on the training leaflet only, then applied to the test leaflet.
3. The network and its training settings are fixed in advance (section 2) and are the same for every window and
   both chains. The test leaflet is not given to `fit` as validation data; it is scored once at the end.
4. Saved, in names the classification step reads:
   - `models_scalers/nn_{chain}_{n}pt_model_ft_last.sav` and `…_scaler_ft_last.sav` (raw: `nn_{chain}_raw_…`)
   - `data_arrays/nn_test_{chain}_{n}pt.pkl` = (X_test, y_test) of the test leaflet, unscaled
   - `models_scalers/metrics/nn_{chain}_{n}pt.json` and a summary table

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import os
import json
import time
import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix)

import keras
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout
from keras.optimizers import Adam

## 2. Settings

In [ ]:
MODEL_PREFIX = "nn"

CHAINS   = ["sn1", "sn2"]
PT_SIZES = ["raw", 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 51, 101, 151, 201]   # must exist in dataframes/sym

TRAIN_LEAFLET = "lower"
TEST_LEAFLET  = "upper"

n_frames     = 4000          # frames per lipid in the raw tables (stop_frame - start_frame)
random_state = 76
N_CV_SPLITS  = 5
RESUME       = True          # skip a (chain, window) whose model is already saved

# Fixed network and training settings, the same for every window and both chains (no tuning, so no CV needed).
# Architecture = the one used in the collaborator's NN_training (34-68-34, ReLU, dropout 0.2, sigmoid output).
NN_LAYERS     = [34, 68, 34]
NN_DROPOUT    = 0.2
LEARNING_RATE = 1e-3
EPOCHS        = 5
BATCH_SIZE    = 256
# The test leaflet is NOT passed as validation data, so nothing about training can depend on it.

main_dir      = "../"
dataframe_dir = os.path.join(main_dir, "heavy_data_NO_GIT/dataframes/sym")
save_dir      = os.path.join(main_dir, "heavy_data_NO_GIT/models_scalers")
data_arrays   = os.path.join(main_dir, "heavy_data_NO_GIT/data_arrays")
metrics_dir   = os.path.join(save_dir, "metrics")
for d in (save_dir, data_arrays, metrics_dir):
    os.makedirs(d, exist_ok=True)

assert TRAIN_LEAFLET != TEST_LEAFLET

## 3. Helper functions

In [ ]:
def suffix_of(pt):
    return "raw" if pt == "raw" else f"{pt}pt"


def frames_per_lipid(pt):
    """Rows per lipid in a table: n_frames for raw, n_frames - (n - 1) after an n-pt running average."""
    return n_frames if pt == "raw" else n_frames - (pt - 1)


def load_table(chain, leaflet, pt):
    path = os.path.join(dataframe_dir, f"training_{chain}_{leaflet}_ft_{suffix_of(pt)}.csv")
    df = pd.read_csv(path)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns="Unnamed: 0")
    assert df.columns[-1] == "label", f"last column of {path} is not 'label'"
    return df.iloc[:, :-1], df.iloc[:, -1].astype(int)


def lipid_ids(n_rows, pt):
    """Tables are lipid-major (each lipid is a block of rows), so lipid id = row // rows-per-lipid."""
    fpl = frames_per_lipid(pt)
    assert n_rows % fpl == 0, f"{n_rows} rows is not a whole number of lipids x {fpl} frames"
    return np.arange(n_rows) // fpl


def evaluate(model, X_scaled, y):
    p = predict_lo(model, X_scaled)                 # P(Lo)
    pred = (p > 0.5).astype(int)
    cm = confusion_matrix(y, pred, labels=[0, 1])
    return {"accuracy": accuracy_score(y, pred),
            "balanced_accuracy": balanced_accuracy_score(y, pred),
            "f1": f1_score(y, pred),
            "roc_auc": roc_auc_score(y, p),
            "ld_called_lo": float(pred[y == 0].mean()),     # false-positive rate (pure DOPC called Lo)
            "lo_called_lo": float(pred[y == 1].mean()),     # true-positive rate (DOPC-60 called Lo)
            "confusion_matrix": cm.tolist()}


def file_names(chain, pt):
    tag = f"{chain}_{suffix_of(pt)}"
    return {"model":  os.path.join(save_dir, f"{MODEL_PREFIX}_{tag}_model_ft_last.sav"),
            "scaler": os.path.join(save_dir, f"{MODEL_PREFIX}_{tag}_scaler_ft_last.sav"),
            "test":   os.path.join(data_arrays, f"{MODEL_PREFIX}_test_{tag}.pkl"),
            "metrics": os.path.join(metrics_dir, f"{MODEL_PREFIX}_{tag}.json")}

In [ ]:
def train_model(X_train_s, y_train, cv_splits=None):
    keras.utils.set_random_seed(random_state)          # same start for every window
    layers = [Input(shape=(X_train_s.shape[1],))]
    for width in NN_LAYERS:
        layers += [Dense(width, activation="relu"), Dropout(NN_DROPOUT)]
    layers += [Dense(1, activation="sigmoid")]
    model = Sequential(layers)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="binary_crossentropy", metrics=["accuracy"])
    hist = model.fit(X_train_s, y_train.to_numpy(), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
    info = {"best_params": {"layers": str(NN_LAYERS), "dropout": NN_DROPOUT, "lr": LEARNING_RATE,
                            "epochs": EPOCHS, "batch_size": BATCH_SIZE},
            "cv_accuracy_train_leaflet": float("nan"),
            "train_accuracy_last_epoch": float(hist.history["accuracy"][-1])}
    return model, info


def predict_lo(model, X_scaled):
    return model.predict(X_scaled, batch_size=4096, verbose=0).reshape(-1)

## 4. Train and test

In [ ]:
def fmt(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m" if h else f"{m}m{s:02d}s"

def is_done(chain, pt):
    n = file_names(chain, pt)
    return RESUME and os.path.exists(n["model"]) and os.path.exists(n["metrics"])

jobs = [(chain, pt) for chain in CHAINS for pt in PT_SIZES]
t_all, n_trained = time.time(), 0
for i, (chain, pt) in enumerate(jobs, start=1):
    tag = f"{chain}_{suffix_of(pt)}"
    names = file_names(chain, pt)
    if is_done(chain, pt):
        print(f"[{i}/{len(jobs)}] skip {tag} (already trained)", flush=True)
        continue
    t = time.time()
    print(f"\n{'=' * 20} [{i}/{len(jobs)}] {tag}: train {TRAIN_LEAFLET}, test {TEST_LEAFLET}  "
          f"(started {time.strftime('%H:%M')}) {'=' * 20}", flush=True)

    # ---- data: one leaflet each ----
    X_train, y_train = load_table(chain, TRAIN_LEAFLET, pt)
    X_test,  y_test  = load_table(chain, TEST_LEAFLET,  pt)
    assert list(X_train.columns) == list(X_test.columns)
    print(f"train rows {len(y_train)} (Lo {y_train.mean():.2f}), test rows {len(y_test)} (Lo {y_test.mean():.2f})")

    # ---- scaler: fitted on the training leaflet only ----
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # ---- CV folds inside the training leaflet, split by whole lipids ----
    groups = lipid_ids(len(y_train), pt)
    assert (pd.Series(y_train.to_numpy()).groupby(groups).nunique() == 1).all(), "a lipid has two labels"
    cv_splits = list(StratifiedGroupKFold(n_splits=N_CV_SPLITS, shuffle=True,
                                          random_state=random_state).split(X_train_s, y_train, groups))

    # ---- train (LR/XGB: hyperparameters chosen by CV on the training leaflet; NN: fixed settings) ----
    model, info = train_model(X_train_s, y_train, cv_splits)

    # ---- test: the other leaflet, scored once ----
    res = evaluate(model, X_test_s, y_test.to_numpy())
    res.update({"tag": tag, "chain": chain, "window": suffix_of(pt),
                "train_leaflet": TRAIN_LEAFLET, "test_leaflet": TEST_LEAFLET,
                "train_rows": int(len(y_train)), "test_rows": int(len(y_test)),
                "minutes": round((time.time() - t) / 60, 2)})
    res.update(info)
    print(f"settings {res['best_params']}  (CV accuracy on {TRAIN_LEAFLET} leaflet {res['cv_accuracy_train_leaflet']:.4f})")
    print(f"TEST ({TEST_LEAFLET} leaflet): accuracy {res['accuracy']:.4f}  balanced {res['balanced_accuracy']:.4f}  "
          f"F1 {res['f1']:.4f}  ROC AUC {res['roc_auc']:.4f}")
    print(f"pure DOPC called Lo {res['ld_called_lo']:.3f}   DOPC-60 called Lo {res['lo_called_lo']:.3f}")
    print("confusion matrix [[Ld->Ld, Ld->Lo], [Lo->Ld, Lo->Lo]]:", res["confusion_matrix"])

    # ---- save ----
    pickle.dump(model,  open(names["model"],  "wb"))
    pickle.dump(scaler, open(names["scaler"], "wb"))
    with open(names["test"], "wb") as f:
        pickle.dump((X_test, y_test), f)
    json.dump(res, open(names["metrics"], "w"), indent=1)
    n_trained += 1
    left = (time.time() - t_all) / n_trained * sum(1 for j in jobs[i:] if not is_done(*j))
    print(f"[{i}/{len(jobs)}] {tag} saved in {fmt(time.time() - t)}  "
          f"(elapsed {fmt(time.time() - t_all)}, about {fmt(left)} left)", flush=True)

print(f"\nall done in {fmt(time.time() - t_all)}")

## 5. Summary table

In [ ]:
rows = []
for chain in CHAINS:
    for pt in PT_SIZES:
        p = file_names(chain, pt)["metrics"]
        if os.path.exists(p):
            rows.append(json.load(open(p)))
summary = pd.DataFrame(rows)[["tag", "train_leaflet", "test_leaflet", "accuracy", "balanced_accuracy", "f1",
                              "roc_auc", "ld_called_lo", "lo_called_lo", "cv_accuracy_train_leaflet", "best_params"]]
summary_path = os.path.join(save_dir, f"{MODEL_PREFIX}_leaflet_split_metrics.txt")
with open(summary_path, "w") as f:
    f.write(f"{MODEL_PREFIX}: train {TRAIN_LEAFLET} leaflet, test {TEST_LEAFLET} leaflet, random_state={random_state}\n\n")
    f.write(summary.to_string(index=False))
print(summary.to_string(index=False))
print("\nsaved", summary_path)